In [1]:
# ======= #
# Imports #                   
# ======= #
import pandas as pd
import numpy as np
import re
import json
import joblib
from pathlib import Path
from collections import Counter
from itertools import combinations

from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy.spatial.distance import cosine, jensenshannon
from collections import defaultdict

from joblib import dump
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
print('Imports ok.')


from su_utils import deserialize_tuple, normalize_to_distribution, primary_from_labels, eval_distribution


Imports ok.


In [2]:
data_dir = Path("/kaggle/input/prepreprocessed-for-bert")

train_ml_export = pd.read_csv(data_dir / "train_ml_export.csv")


val_ml_export   = pd.read_csv(data_dir / "val_ml_export.csv")
#DEBUGGING
#val_ml_export_debug = pd.read_csv(data_dir / "val_ml_export.csv")
#deserialize back to tuples
val_ml_export["labels_uo"] = val_ml_export["labels_uo"].apply(deserialize_tuple)
val_ml_export["labels_pct"] = val_ml_export["labels_pct"].apply(deserialize_tuple)

label_list      = joblib.load(data_dir / "uo_label_list.joblib")
print ("dirs ok")

data_dir = Path("/kaggle/input/lis070-su-admin-data")
model_dir = Path("/kaggle/input/kbbert-swe/kbbert_swe")


dirs ok


In [3]:
# Identify label columns
label_cols = [c for c in train_ml_export.columns if c.startswith("y_")]
num_labels = len(label_cols)

train_texts = train_ml_export["text"].astype(str).tolist()
val_texts   = val_ml_export["text"].astype(str).tolist()

Y_train = train_ml_export[label_cols].values.astype("float32")
Y_val   = val_ml_export[label_cols].values.astype("float32")
print("Done")

Done


In [4]:
#Build HF dataset
from datasets import Dataset

train_hf = Dataset.from_dict({"text": train_texts, "labels": list(Y_train)})
val_hf   = Dataset.from_dict({"text": val_texts, "labels": list(Y_val)})
print("Done")

Done


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = model_dir
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

def tokenize_batch(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=512,
    )
    enc["labels"] = batch["labels"]
    return enc

train_tokenized = train_hf.map(tokenize_batch, batched=True)
val_tokenized   = val_hf.map(tokenize_batch, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    local_files_only=True,
    ignore_mismatched_sizes=True,  # FIX 1
)
print("Done")

Map:   0%|          | 0/7865 [00:00<?, ? examples/s]

Map:   0%|          | 0/1905 [00:00<?, ? examples/s]

2025-12-10 22:48:12.822231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765406892.987303      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765406893.033778      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/kbbert-swe/kbbert_swe and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Done


In [6]:
#move to util file
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # labels will come in as shape (batch, num_labels)
    preds = (1 / (1 + np.exp(-logits)) >= 0.5).astype(int)

    # subset accuracy = exact match of all labels
    subset_acc = accuracy_score(labels, preds)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    hamming = hamming_loss(labels, preds)

    return {
        "subset_accuracy": subset_acc,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamming,
    }

In [7]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./kbbert_uo_multilabel",
    report_to="none",  # Disisable wandb
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics

Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro F1,Macro F1,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.074400,0.074564,0.885564,0.921751,0.881125,0.018583,34.021700,55.994000,1.764000
2,0.041500,0.056693,0.899738,0.931224,0.900843,0.016430,34.518900,55.187000,1.738000
3,0.020900,0.053967,0.907612,0.935470,0.904603,0.015433,33.815500,56.335000,1.774000


{'eval_loss': 0.05396690219640732,
 'eval_subset_accuracy': 0.9076115485564304,
 'eval_micro_f1': 0.9354697102721685,
 'eval_macro_f1': 0.9046030552413425,
 'eval_hamming_loss': 0.015433070866141733,
 'eval_runtime': 34.2307,
 'eval_samples_per_second': 55.652,
 'eval_steps_per_second': 1.753,
 'epoch': 3.0}

In [8]:
# 1) Get raw logits from the model on the validation set
pred_out = trainer.predict(val_tokenized)
logits = pred_out.predictions              # shape: (n_val, num_labels)

# 2) Convert logits -> probabilities via sigmoid
Y_prob = 1 / (1 + np.exp(-logits))        # same as torch.sigmoid but in NumPy



dist_preds = normalize_to_distribution(Y_prob, scale=100)
dist_preds.shape


(1905, 10)

In [9]:
# sanity check
dist_preds[:5], dist_preds[:5].sum(axis=1)

(array([[ 0.8681492 ,  0.54466534,  0.565417  ,  0.86883736, 94.6426    ,
          0.6491409 ,  0.34952497,  0.36379495,  0.67338675,  0.47447878],
        [ 0.83884215,  0.49501625,  0.55023074,  0.796719  , 94.77965   ,
          0.6885804 ,  0.33133683,  0.35883537,  0.6833666 ,  0.477427  ],
        [ 0.84661937,  0.5342119 ,  0.5774648 ,  0.8284516 , 94.72741   ,
          0.69698346,  0.34854773,  0.34859097,  0.63112164,  0.4605909 ],
        [ 0.861181  ,  0.5370808 ,  0.5806619 ,  0.7963018 , 94.75254   ,
          0.69686604,  0.33983693,  0.3479931 ,  0.6329046 ,  0.45463607],
        [ 0.829264  ,  0.49644396,  0.54961574,  0.9106305 , 94.57316   ,
          0.7381986 ,  0.34359932,  0.3620862 ,  0.6924665 ,  0.50452673]],
       dtype=float32),
 array([ 99.99999, 100.     ,  99.99999, 100.     ,  99.99999],
       dtype=float32))

In [10]:
val_ml_export.columns

Index(['id', 'text', 'labels_uo', 'labels_pct', 'y_2434', 'y_2436', 'y_2438',
       'y_2439', 'y_2441', 'y_2442', 'y_2444', 'y_2445', 'y_2447', 'y_2451'],
      dtype='object')

In [11]:
val_ml_export.head

<bound method NDFrame.head of          id  \
0      6670   
1      6670   
2      6672   
3      6672   
4      6848   
...     ...   
1900  49927   
1901  50026   
1902  50077   
1903  50163   
1904  50202   

                                                                                                                         text  \
0     Examensarbete i molekylära livsvetenskaper Kursen består av ett teoretiskt eller praktiskt arbete som utformas indiv...   
1     Examensarbete i molekylära livsvetenskaper a. Kursen består av ett inledande moment där en detaljerad projektplan ut...   
2     Examensarbete i molekylärbiologi Kursen behandlar informationssökning, upphovsrätt och plagiat, vetenskapligt och po...   
3     Examensarbete i molekylärbiologi Kursen behandlar informationssökning, upphovsrätt och plagiat, vetenskapligt och po...   
4     Gener, celler och populationer a. Kursen behandlar grundläggande cellbiologi, molekylärbiologi, genetik och mikrobio...   
...            

In [12]:
#new
def make_gold_distribution(row, uo_to_idx, num_labels):
    """
    Convert (labels_uo, labels_pct) tuples into a distribution vector.
    
    Args:
        row: DataFrame row with 'labels_uo' and 'labels_pct' columns
        uo_to_idx: dict mapping UO codes to indices (e.g., {2434: 0, 2436: 1, ...})
        num_labels: total number of labels
    
    Returns:
        np.array of shape (num_labels,) with percentages (summing to 100)
    """
    dist = np.zeros(num_labels, dtype=float)
    uos  = row["labels_uo"]
    pcts = row["labels_pct"]
    
    if not isinstance(uos, (list, tuple)) or len(uos) == 0:
        return dist
    
    # Handle case where pcts might be missing or mismatched
    if not isinstance(pcts, (list, tuple)) or len(pcts) != len(uos):
        # Fallback: equal distribution among present labels
        equal_pct = 100.0 / len(uos)
        pcts = [equal_pct] * len(uos)
    
    for uo, pct in zip(uos, pcts):
        if uo in uo_to_idx:
            dist[uo_to_idx[uo]] = pct
    
    # Normalize to exactly 100 if there's rounding error
    total = dist.sum()
    if total > 0 and abs(total - 100) > 0.01:
        dist = dist * (100.0 / total)
    
    return dist


# Build the mapping from UO codes to indices
# label_list is loaded from joblib: e.g., [2434, 2436, 2438, ...]
uo_to_idx = {uo: i for i, uo in enumerate(label_list)}
num_labels = len(label_list)

# Build gold distribution matrix
gold_dist = np.vstack([
    make_gold_distribution(row, uo_to_idx, num_labels)
    for _, row in val_ml_export.iterrows()
])

print(f"Gold distribution shape: {gold_dist.shape}")
print(f"Sample gold dist (first 3 rows):\n{gold_dist[:3]}")

Gold distribution shape: (1905, 10)
Sample gold dist (first 3 rows):
[[  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]]


In [13]:
print(f"Sample gold dist (first 3 rows):\n{gold_dist[:20]}")

Sample gold dist (first 3 rows):
[[  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.  

In [14]:
from su_utils import normalize_to_distribution
pred_dist = normalize_to_distribution(Y_prob, scale=100)

print(f"Pred distribution shape: {pred_dist.shape}")
print(f"Sample pred dist (first 3 rows):\n{pred_dist[:3]}")

Pred distribution shape: (1905, 10)
Sample pred dist (first 3 rows):
[[ 0.8681492   0.54466534  0.565417    0.86883736 94.6426      0.6491409
   0.34952497  0.36379495  0.67338675  0.47447878]
 [ 0.83884215  0.49501625  0.55023074  0.796719   94.77965     0.6885804
   0.33133683  0.35883537  0.6833666   0.477427  ]
 [ 0.84661937  0.5342119   0.5774648   0.8284516  94.72741     0.69698346
   0.34854773  0.34859097  0.63112164  0.4605909 ]]


In [15]:
from scipy.spatial.distance import cosine, jensenshannon


def eval_distrib(gold_dist, pred_dist):
    """
    Compute distributional similarity metrics between gold and predicted distributions.
    
    Args:
        gold_dist: array of shape (n_samples, n_labels), percentages summing to 100
        pred_dist: array of shape (n_samples, n_labels), percentages summing to 100
    
    Returns:
        dict with metrics
    """
    n_samples = len(gold_dist)
    
    # Convert to probability distributions (sum to 1) for some metrics
    gold_prob = gold_dist / 100.0
    pred_prob = pred_dist / 100.0
    
    # --- Mean Absolute Error (on percentages) ---
    mae = np.abs(gold_dist - pred_dist).mean()
    
    # --- Per-sample MAE ---
    per_sample_mae = np.abs(gold_dist - pred_dist).mean(axis=1)
    
    # --- Cosine Similarity (per sample, then average) ---
    cosine_sims = []
    for g, p in zip(gold_dist, pred_dist):
        if g.sum() > 0 and p.sum() > 0:
            cosine_sims.append(1 - cosine(g, p))  # cosine() returns distance
        else:
            cosine_sims.append(0.0)
    mean_cosine = np.mean(cosine_sims)
    
    # --- Jensen-Shannon Divergence (per sample, then average) ---
    js_divs = []
    for g, p in zip(gold_prob, pred_prob):
        # Add small epsilon to avoid log(0)
        g_safe = g + 1e-10
        p_safe = p + 1e-10
        g_safe = g_safe / g_safe.sum()
        p_safe = p_safe / p_safe.sum()
        js_divs.append(jensenshannon(g_safe, p_safe))
    mean_js = np.mean(js_divs)
    
    # --- Top-1 Accuracy (does the highest predicted label match highest gold label?) ---
    gold_top1 = np.argmax(gold_dist, axis=1)
    pred_top1 = np.argmax(pred_dist, axis=1)
    top1_acc = (gold_top1 == pred_top1).mean()
    
    return {
        "mae_pct": mae,
        "mean_cosine_sim": mean_cosine,
        "mean_js_divergence": mean_js,
        "top1_accuracy": top1_acc,
        "per_sample_mae": per_sample_mae,
    }

In [ ]:
from scipy.spatial.distance import cosine, jensenshannon


# Run evaluation

dist_metrics = eval_distrib(gold_dist, pred_dist)

print("\n" + "="*50)
print("DISTRIBUTIONAL EVALUATION METRICS")
print("="*50)
print(f"Mean Absolute Error (percentage points): {dist_metrics['mae_pct']:.2f}")
print(f"Mean Cosine Similarity:                  {dist_metrics['mean_cosine_sim']:.4f}")
print(f"Mean Jensen-Shannon Divergence:          {dist_metrics['mean_js_divergence']:.4f}")
print(f"Top-1 Accuracy (primary label match):    {dist_metrics['top1_accuracy']:.4f}")
print(f"JSD: {dist_metrics['mean_js_divergence']:.4f}") #added JSD 2025-12-31


DISTRIBUTIONAL EVALUATION METRICS
Mean Absolute Error (percentage points): 2.79
Mean Cosine Similarity:                  0.9382
Mean Jensen-Shannon Divergence:          0.1915
Top-1 Accuracy (primary label match):    0.8525


In [17]:
# DETAILED ANALYSIS: Per-Label MAE

per_label_mae = np.abs(gold_dist - pred_dist).mean(axis=0)
label_mae_df = pd.DataFrame({
    "uo_code": label_list,
    "mae_pct": per_label_mae
}).sort_values("mae_pct", ascending=False)

print("\nPer-Label MAE (percentage points):")
print(label_mae_df.to_string(index=False))


Per-Label MAE (percentage points):
 uo_code  mae_pct
    2442 7.022961
    2434 3.773409
    2441 3.452049
    2438 2.910137
    2447 2.828696
    2444 2.301982
    2439 1.794413
    2445 1.479483
    2436 1.377615
    2451 0.960420


In [18]:
# ERROR ANALYSIS: Worst predictions

per_sample_mae = dist_metrics["per_sample_mae"]
worst_idx = np.argsort(per_sample_mae)[-10:][::-1]  # top 10 worst

print("\n" + "="*50)
print("WORST PREDICTIONS (highest MAE)")
print("="*50)
for idx in worst_idx:
    row = val_ml_export.iloc[idx]
    print(f"\nCourse ID: {row['id']}")
    print(f"  Gold:      {dict(zip(row['labels_uo'], row['labels_pct']))}")
    print(f"  Gold dist: {gold_dist[idx]}")
    print(f"  Pred dist: {pred_dist[idx].round(1)}")
    print(f"  MAE:       {per_sample_mae[idx]:.2f}")


WORST PREDICTIONS (highest MAE)

Course ID: 31197
  Gold:      {2447: 100.0}
  Gold dist: [  0.   0.   0.   0.   0.   0.   0.   0. 100.   0.]
  Pred dist: [96.5  0.3  0.4  0.3  0.6  0.4  0.3  0.3  0.5  0.3]
  MAE:       19.91

Course ID: 14747
  Gold:      {2438: 100.0}
  Gold dist: [  0.   0. 100.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [96.8  0.3  0.5  0.2  0.4  0.7  0.3  0.2  0.4  0.3]
  MAE:       19.90

Course ID: 14747
  Gold:      {2438: 100.0}
  Gold dist: [  0.   0. 100.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [96.8  0.3  0.5  0.2  0.5  0.6  0.3  0.2  0.4  0.3]
  MAE:       19.90

Course ID: 49441
  Gold:      {2436: 100.0}
  Gold dist: [  0. 100.   0.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [25.9  0.6  0.4  0.3  0.2 71.4  0.2  0.3  0.4  0.2]
  MAE:       19.88

Course ID: 43983
  Gold:      {2434: 100.0}
  Gold dist: [100.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [ 0.6  0.5  0.4 45.6  0.6 49.8  1.1  0.4  0.5  0.5]
  MAE:       19.

## Extended Analysis

In [19]:
UO_NAMES = {
    2434: "HU",  # Humaniora
    2436: "JU",  # Juridik
    2438: "LU",  # Lärarutbildning  
    2439: "ME",  # Medicin
    2441: "NA",  # Naturvetenskap
    2442: "SA",  # Samhällsvetenskap
    2444: "TE",  # Teknik
    2445: "VÅ",  # Vård
    2447: "ÖV",  # Övrigt
    2451: "VU"   # Verksamhetsförlagd utbildning
}

def get_uo_name(code):
    """Get UO name from code, with fallback."""
    return UO_NAMES.get(code, f"UO_{code}")

# Add names to per-label MAE
label_mae_df["uo_name"] = label_mae_df["uo_code"].apply(get_uo_name)
print("Per-Label MAE with names:")
print(label_mae_df[["uo_code", "uo_name", "mae_pct"]].to_string(index=False))

Per-Label MAE with names:
 uo_code uo_name  mae_pct
    2442      SA 7.022961
    2434      HU 3.773409
    2441      NA 3.452049
    2438      LU 2.910137
    2447      ÖV 2.828696
    2444      TE 2.301982
    2439      ME 1.794413
    2445      VÅ 1.479483
    2436      JU 1.377615
    2451      VU 0.960420


In [20]:
# =============================================================================
# 2. CONFUSION ANALYSIS: Which labels get confused with which?
# =============================================================================

def analyze_confusion(gold_dist, pred_dist, label_list, threshold=10):
    """
    Analyze systematic confusion between labels.
    
    Returns DataFrame showing: when gold is X, model often predicts Y instead.
    threshold: minimum average percentage points to report
    """
    n_labels = len(label_list)
    
    # For each sample, find gold primary and pred primary
    gold_primary = np.argmax(gold_dist, axis=1)
    pred_primary = np.argmax(pred_dist, axis=1)
    
    # Build confusion matrix (rows=gold, cols=pred)
    confusion = np.zeros((n_labels, n_labels))
    counts = np.zeros(n_labels)
    
    for g, p in zip(gold_primary, pred_primary):
        confusion[g, p] += 1
        counts[g] += 1
    
    # Normalize to percentages (row-wise)
    with np.errstate(divide='ignore', invalid='ignore'):
        confusion_pct = (confusion.T / counts).T * 100
        confusion_pct = np.nan_to_num(confusion_pct)
    
    # Find significant confusions (off-diagonal)
    confusions = []
    for i in range(n_labels):
        for j in range(n_labels):
            if i != j and confusion_pct[i, j] >= threshold:
                confusions.append({
                    "gold_code": label_list[i],
                    "gold_name": get_uo_name(label_list[i]),
                    "pred_code": label_list[j],
                    "pred_name": get_uo_name(label_list[j]),
                    "confusion_pct": confusion_pct[i, j],
                    "n_cases": int(confusion[i, j]),
                })
    
    return pd.DataFrame(confusions).sort_values("confusion_pct", ascending=False)

confusion_df = analyze_confusion(gold_dist, pred_dist, label_list, threshold=5)
print("\n" + "="*60)
print("LABEL CONFUSION ANALYSIS")
print("(When gold is X, model predicts Y instead)")
print("="*60)
if len(confusion_df) > 0:
    print(confusion_df.to_string(index=False))
else:
    print("No significant confusions above threshold.")


LABEL CONFUSION ANALYSIS
(When gold is X, model predicts Y instead)
 gold_code gold_name  pred_code pred_name  confusion_pct  n_cases
      2444        TE       2442        SA      94.594595       70
      2445        VÅ       2442        SA      85.185185       23
      2439        ME       2442        SA      84.507042       60
      2445        VÅ       2438        LU      14.814815        4
      2438        LU       2442        SA      12.328767        9


In [21]:
#=============================================================================
# 3. DETAILED ERROR INSPECTION WITH TEXT
# =============================================================================

def get_worst_predictions_detailed(val_df, gold_dist, pred_dist, per_sample_mae, 
                                    label_list, n=20, text_preview_len=300):
    """
    Get detailed info on worst predictions including course text.
    """
    worst_idx = np.argsort(per_sample_mae)[-n:][::-1]
    
    rows = []
    for idx in worst_idx:
        row = val_df.iloc[idx]
        
        # Gold info
        gold_uos = row["labels_uo"]
        gold_pcts = row["labels_pct"]
        gold_primary_idx = np.argmax(gold_dist[idx])
        gold_primary_code = label_list[gold_primary_idx]
        
        # Pred info
        pred_primary_idx = np.argmax(pred_dist[idx])
        pred_primary_code = label_list[pred_primary_idx]
        pred_primary_pct = pred_dist[idx][pred_primary_idx]
        
        # Text preview
        text = row.get("text", "")
        if len(text) > text_preview_len:
            text_preview = text[:text_preview_len] + "..."
        else:
            text_preview = text
        
        rows.append({
            "id": row["id"],
            "mae": per_sample_mae[idx],
            "gold_labels": dict(zip(gold_uos, gold_pcts)) if gold_uos else {},
            "gold_primary": f"{gold_primary_code} ({get_uo_name(gold_primary_code)})",
            "pred_primary": f"{pred_primary_code} ({get_uo_name(pred_primary_code)})",
            "pred_primary_pct": f"{pred_primary_pct:.1f}%",
            "correct_primary": gold_primary_code == pred_primary_code,
            "text_preview": text_preview,
        })
    
    return pd.DataFrame(rows)

worst_detailed = get_worst_predictions_detailed(
    val_ml_export, gold_dist, pred_dist, per_sample_mae, label_list, n=15
)

print("\n" + "="*60)
print("WORST PREDICTIONS - DETAILED VIEW")
print("="*60)
for _, row in worst_detailed.iterrows():
    print(f"\n{'─'*60}")
    print(f"Course ID: {row['id']} | MAE: {row['mae']:.2f}")
    print(f"Gold: {row['gold_labels']} → {row['gold_primary']}")
    print(f"Pred: {row['pred_primary']} ({row['pred_primary_pct']})")
    print(f"Primary correct: {'✓' if row['correct_primary'] else '✗'}")
    print(f"Text: {row['text_preview']}")


WORST PREDICTIONS - DETAILED VIEW

────────────────────────────────────────────────────────────
Course ID: 31197 | MAE: 19.91
Gold: {2447: 100.0} → 2447 (ÖV)
Pred: 2434 (HU) (96.5%)
Primary correct: ✗
Text: Tolkning – masterkurs Studenten genomför ett examensarbete som utformas som en översättningsvetenskaplig forskningsuppgift. I kursen ingår handlett examensarbete och instruktion i akademiskt skrivande och information om den forskning som bedrivs på institutionen. Dessutom ingår schemalagda seminarie...

────────────────────────────────────────────────────────────
Course ID: 14747 | MAE: 19.90
Gold: {2438: 100.0} → 2438 (LU)
Pred: 2434 (HU) (96.8%)
Primary correct: ✗
Text: Självständigt arbete I kursen ingår att självständigt genomföra ett språk- eller litteraturdidaktiskt forsknings- eller utvecklingsarbete med relevans för lärarprofessionen. Ämne väljs i samråd med den handledare som utsetts av institutionen. I arbetet ingår en kritisk reflektion över vetenskapliga ...

──────────

In [22]:
# =============================================================================
# 4. EXPORT CSV FOR EXPERT REVIEW
# =============================================================================

def create_expert_review_csv(val_df, gold_dist, pred_dist, per_sample_mae,
                              label_list, output_path, n_cases=50):
    """
    Create a CSV file for expert review with:
    - Worst predictions by MAE
    - Edge cases (high uncertainty)
    - Random sample for baseline
    """
    n_samples = len(val_df)
    
    # Get indices for different categories
    worst_idx = np.argsort(per_sample_mae)[-n_cases//2:][::-1]  # Worst MAE
    
    # High uncertainty: pred distribution is relatively flat (high entropy)
    pred_entropy = -np.sum(pred_dist/100 * np.log(pred_dist/100 + 1e-10), axis=1)
    uncertain_idx = np.argsort(pred_entropy)[-n_cases//4:][::-1]
    
    # Random sample
    np.random.seed(42)
    random_idx = np.random.choice(n_samples, size=n_cases//4, replace=False)
    
    # Combine and deduplicate
    all_idx = list(dict.fromkeys(list(worst_idx) + list(uncertain_idx) + list(random_idx)))
    all_idx = all_idx[:n_cases]
    
    rows = []
    for idx in all_idx:
        row = val_df.iloc[idx]
        
        # Basic info
        gold_uos = row["labels_uo"] 
        gold_pcts = row["labels_pct"]
        
        # Primary labels
        gold_primary_idx = np.argmax(gold_dist[idx])
        pred_primary_idx = np.argmax(pred_dist[idx])
        gold_primary_code = label_list[gold_primary_idx]
        pred_primary_code = label_list[pred_primary_idx]
        
        # Selection reason
        if idx in worst_idx:
            selection_reason = "high_mae"
        elif idx in uncertain_idx:
            selection_reason = "high_uncertainty"
        else:
            selection_reason = "random"
        
        record = {
            "id": row["id"],
            "name": row.get("name", ""),
            "text": row.get("text", ""),
            "selection_reason": selection_reason,
            "mae": round(per_sample_mae[idx], 2),
            "pred_entropy": round(pred_entropy[idx], 4),
            # Gold info
            "primary_uo": gold_primary_code,
            "primary_uo_name": get_uo_name(gold_primary_code),
            "labels_uo": str(list(gold_uos)) if gold_uos else "[]",
            "labels_pct": str(list(gold_pcts)) if gold_pcts else "[]",
            # Prediction info
            "pred_primary_uo": pred_primary_code,
            "pred_primary_uo_name": get_uo_name(pred_primary_code),
            "pred_primary_pct": round(pred_dist[idx][pred_primary_idx], 1),
            "primary_match": gold_primary_code == pred_primary_code,
        }
        
        # Add per-label predictions
        for i, code in enumerate(label_list):
            record[f"pred_{code}"] = round(pred_dist[idx][i], 2)
            record[f"gold_{code}"] = round(gold_dist[idx][i], 2)
        
        rows.append(record)
    
    df_export = pd.DataFrame(rows)
    
    # Sort by MAE descending
    df_export = df_export.sort_values("mae", ascending=False)
    
    # Save
    df_export.to_csv(output_path, index=False)
    print(f"\nExported {len(df_export)} cases to {output_path}")
    
    # Summary stats
    print(f"\nExport summary:")
    print(f"  - High MAE cases: {(df_export['selection_reason'] == 'high_mae').sum()}")
    print(f"  - High uncertainty cases: {(df_export['selection_reason'] == 'high_uncertainty').sum()}")
    print(f"  - Random cases: {(df_export['selection_reason'] == 'random').sum()}")
    print(f"  - Primary match rate: {df_export['primary_match'].mean():.1%}")
    
    return df_export

In [23]:
# Create the expert review CSV
expert_df = create_expert_review_csv(
    val_ml_export, 
    gold_dist, 
    pred_dist, 
    per_sample_mae,
    label_list,
    output_path="bert_expert_review_cases.csv",  # Adjust path as needed
    n_cases=50
)

print("\n" + "="*60)
print("SAMPLE OF EXPERT REVIEW EXPORT")
print("="*60)
print(expert_df[["id", "mae", "primary_uo_name", "pred_primary_uo_name", "primary_match"]].head(10))


Exported 49 cases to bert_expert_review_cases.csv

Export summary:
  - High MAE cases: 25
  - High uncertainty cases: 13
  - Random cases: 11
  - Primary match rate: 22.4%

SAMPLE OF EXPERT REVIEW EXPORT
      id    mae primary_uo_name pred_primary_uo_name  primary_match
0  31197  19.91              ÖV                   HU          False
1  14747  19.90              LU                   HU          False
2  14747  19.90              LU                   HU          False
3  49441  19.88              JU                   SA          False
4  43983  19.87              HU                   SA          False
5  43983  19.87              HU                   SA          False
6  26238  19.82              NA                   HU          False
7  15798  19.82              ÖV                   HU          False
8  15798  19.82              ÖV                   HU          False
9  26238  19.80              NA                   HU          False


In [24]:
# =============================================================================
# 5. SUMMARY STATISTICS
# =============================================================================

print("\n" + "="*60)
print("OVERALL SUMMARY")
print("="*60)

# Label distribution in gold vs predicted primaries
gold_primary = np.argmax(gold_dist, axis=1)
pred_primary = np.argmax(pred_dist, axis=1)

print("\nPrimary label distribution (Gold vs Predicted):")
for i, code in enumerate(label_list):
    gold_count = (gold_primary == i).sum()
    pred_count = (pred_primary == i).sum()
    print(f"  {get_uo_name(code):25s}: Gold={gold_count:4d}, Pred={pred_count:4d}, Diff={pred_count-gold_count:+4d}")

# Accuracy by label
print("\nPer-label primary accuracy:")
for i, code in enumerate(label_list):
    mask = gold_primary == i
    if mask.sum() > 0:
        acc = (pred_primary[mask] == i).mean()
        print(f"  {get_uo_name(code):25s}: {acc:.1%} ({mask.sum()} cases)")



OVERALL SUMMARY

Primary label distribution (Gold vs Predicted):
  HU                       : Gold= 677, Pred= 671, Diff=  -6
  JU                       : Gold=  77, Pred=  76, Diff=  -1
  LU                       : Gold=  73, Pred=  87, Diff= +14
  ME                       : Gold=  71, Pred=  10, Diff= -61
  NA                       : Gold= 321, Pred= 316, Diff=  -5
  SA                       : Gold= 439, Pred= 588, Diff=+149
  TE                       : Gold=  74, Pred=   0, Diff= -74
  VÅ                       : Gold=  27, Pred=   0, Diff= -27
  ÖV                       : Gold= 123, Pred= 135, Diff= +12
  VU                       : Gold=  23, Pred=  22, Diff=  -1

Per-label primary accuracy:
  HU                       : 96.5% (677 cases)
  JU                       : 93.5% (77 cases)
  LU                       : 82.2% (73 cases)
  ME                       : 14.1% (71 cases)
  NA                       : 91.9% (321 cases)
  SA                       : 90.9% (439 cases)
  TE            

In [25]:
# =============================================================================
# SAVE PREDICTIONS FOR BOOTSTRAP TESTING
# =============================================================================

# Binary predictions (thresholded at 0.5)
Y_pred_bert_binary = (Y_prob >= 0.5).astype(int)

# Save for bootstraping:
np.save("/kaggle/working/bert_binary_Y_pred.npy", Y_pred_bert_binary)  # For multi-label metrics
np.save("/kaggle/working/bert_binary_Y_prob.npy", Y_prob)              # Raw sigmoid probs
np.save("/kaggle/working/bert_binary_pred_dist.npy", pred_dist)        # Normalized to 100

print(f"✓ Saved Y_pred_bert_binary: {Y_pred_bert_binary.shape}")
print(f"✓ Saved Y_prob (sigmoid): {Y_prob.shape}")
print(f"✓ Saved pred_dist (normalized): {pred_dist.shape}")

✓ Saved Y_pred_bert_binary: (1905, 10)
✓ Saved Y_prob (sigmoid): (1905, 10)
✓ Saved pred_dist (normalized): (1905, 10)
